# Day 051 — Exercise 1: Session State

**What you'll build:** `init_session(state)`, `add_message(state, role, content)`, and `reset_messages(state)` — the functions that manage a Streamlit-style `st.session_state` dict for a chat app.

**Why it matters:** Streamlit reruns your *entire script* top-to-bottom on every click, keystroke, or slider drag. The only thing that survives a rerun is `st.session_state` — a plain dict. So your initialisation must be **idempotent**: set a key only if it's missing, or every rerun would wipe the conversation. That single rule is the heart of Streamlit state.

## Provided: Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import ollama

## Your Implementation

In [ ]:
def init_session(state: dict) -> dict:
    """
    Idempotently initialise a session_state dict. Only set a key if ABSENT.
    Ensure: 'messages' (empty list) and 'settings'
    (model='llama3.2', temperature=0.7, system_prompt='You are a helpful assistant.').
    """
    # TODO: if 'messages' not in state: state['messages'] = []
    # TODO: if 'settings' not in state: state['settings'] = {..model, temperature, system_prompt..}
    return state


def add_message(state: dict, role: str, content: str) -> dict:
    """Append {'role', 'content'} to state['messages']; reject bad roles; return it."""
    # TODO: if role not in ('user', 'assistant', 'system'): raise ValueError(...)
    # TODO: msg = {'role': role, 'content': content}
    # TODO: state['messages'].append(msg); return msg
    pass


def reset_messages(state: dict) -> None:
    """Clear state['messages'] but keep state['settings']."""
    # TODO: state['messages'] = []
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: init_session returns a dict with 'messages' and 'settings'
    try:
        st_state = {}
        out = init_session(st_state)
        assert isinstance(out, dict), f'expected dict, got {type(out).__name__}'
        assert 'messages' in out and 'settings' in out, 'missing messages/settings'
        assert out['messages'] == [], 'messages should start empty'
        passed += 1; print('✅ Check 1: init_session sets messages + settings')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: init_session is IDEMPOTENT (does not wipe existing data)
    try:
        st_state = {}
        init_session(st_state)
        st_state['messages'].append({'role': 'user', 'content': 'hi'})
        init_session(st_state)  # rerun simulation
        assert len(st_state['messages']) == 1, 'idempotent init must NOT clear messages'
        passed += 1; print('✅ Check 2: init_session is idempotent across reruns')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: add_message appends and returns the message
    try:
        st_state = init_session({})
        msg = add_message(st_state, 'user', 'hello')
        assert msg == {'role': 'user', 'content': 'hello'}, f'bad message: {msg}'
        assert st_state['messages'][-1] == msg, 'message not appended'
        passed += 1; print('✅ Check 3: add_message appends + returns the message')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: add_message rejects an invalid role
    try:
        st_state = init_session({})
        raised = False
        try:
            add_message(st_state, 'robot', 'nope')
        except ValueError:
            raised = True
        assert raised, 'expected ValueError on invalid role'
        passed += 1; print('✅ Check 4: add_message rejects invalid roles')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: reset_messages clears messages but keeps settings
    try:
        st_state = init_session({})
        add_message(st_state, 'user', 'a')
        add_message(st_state, 'assistant', 'b')
        reset_messages(st_state)
        assert st_state['messages'] == [], 'messages not cleared'
        assert 'settings' in st_state, 'settings should survive a reset'
        passed += 1; print('✅ Check 5: reset_messages clears messages, keeps settings')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def init_session(state: dict) -> dict:
    """
    Idempotently initialise a Streamlit-style session_state dict.

    Streamlit reruns the WHOLE script top-to-bottom on every interaction, so
    initialisation must never overwrite existing data. Only set a key if absent.

    Ensures keys:
        'messages'  -> list of {'role', 'content'} dicts (starts empty)
        'settings'  -> {'model', 'temperature', 'system_prompt'}
    Returns the same dict, mutated in place.
    """
    if 'messages' not in state:
        state['messages'] = []
    if 'settings' not in state:
        state['settings'] = {
            'model': 'llama3.2',
            'temperature': 0.7,
            'system_prompt': 'You are a helpful assistant.',
        }
    return state


def add_message(state: dict, role: str, content: str) -> dict:
    """Append a {'role', 'content'} message to state['messages']; return it."""
    if role not in ('user', 'assistant', 'system'):
        raise ValueError(f'invalid role: {role!r}')
    msg = {'role': role, 'content': content}
    state['messages'].append(msg)
    return msg


def reset_messages(state: dict) -> None:
    """Clear the conversation but keep settings (a 'Clear chat' button)."""
    state['messages'] = []
```

**Why this works:** The `if key not in state` guard is what makes init safe to call on every rerun — Streamlit runs the whole script each time, so a naive `state['messages'] = []` would erase the chat on every interaction. `add_message` validates the role up front (fail fast) so bad data never reaches the model. `reset_messages` rebinds only `messages`, leaving `settings` intact — exactly what a 'Clear chat' button should do.
</details>